In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, LongType

spark = (SparkSession.builder
    .appName("Churn_Dim_Customer_ETL")
    .master("local[*]")
    .enableHiveSupport()
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")


In [2]:
BASE = "hdfs:///churn_proj"

# Customers PreProcessing

In [3]:
df_customers = (
    spark.read
    .option("header", True)
    .csv(f"{BASE}/customers")
)

print("Rows:", df_customers.count())
df_customers.printSchema()

Rows: 10150
root
 |-- row_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- geography: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- tenure_months: string (nullable = true)
 |-- date_opened: string (nullable = true)
 |-- credit_score: string (nullable = true)
 |-- balance: string (nullable = true)
 |-- num_products: string (nullable = true)
 |-- has_cr_card: string (nullable = true)
 |-- estimated_salary: string (nullable = true)
 |-- is_churned: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_batch_ts: string (nullable = true)



* All Features has `string` data type

In [4]:
df_customers.show(10, truncate=False)

+------+-----------+----+------+---------+---------+-------------+-----------+------------+---------+------------+-----------+----------------+----------+-------------+---------------------+
|row_id|customer_id|age |gender|geography|is_active|tenure_months|date_opened|credit_score|balance  |num_products|has_cr_card|estimated_salary|is_churned|source_file  |load_batch_ts        |
+------+-----------+----+------+---------+---------+-------------+-----------+------------+---------+------------+-----------+----------------+----------+-------------+---------------------+
|2001  |15613656   |58.0|Male  |France   |1        |12           |2025-09-01 |842         |63492.94 |1           |1          |83172.19        |0         |customers.csv|2026-09-02 02:28:03.0|
|2002  |15734311   |27.0|Female|France   |1        |36           |2023-09-02 |661         |0.0      |2           |1          |76889.79        |0         |customers.csv|2026-09-02 02:28:03.0|
|2003  |15657214   |74.0|Male  |France   |1  

In [5]:
df_customers = df_customers.drop("row_id")

### `row_id` — Removed

`row_id` was only used as a technical identifier and is not needed for the final customer dataset, so it was removed.

In [6]:
# Trim whitespace from all columns (Now All string DataType)

cols = df_customers.columns

for c in cols:
    df_customers = df_customers.withColumn(
        c,
        F.trim(F.col(c))
    )

## `null` , `N/A` , `unkown` Values require to be unified 

In [7]:
# Convert sentinel values to NULL
for c in cols:
    df_customers = df_customers.withColumn(
        c,
        F.when(
            F.upper(F.trim(F.col(c))).isin(
                "N/A",
                "UNKNOWN",
                "NULL"
            ),
            None
        )
        .otherwise(F.col(c))
    )

# Check for dupilcates 

In [8]:
df_customers.groupBy(df_customers.columns) \
    .count() \
    .filter(F.col("count") > 1) \
    .show(truncate=False)

+-----------+----+------+---------+---------+-------------+-----------+------------+---------+------------+-----------+----------------+----------+-------------+---------------------+-----+
|customer_id|age |gender|geography|is_active|tenure_months|date_opened|credit_score|balance  |num_products|has_cr_card|estimated_salary|is_churned|source_file  |load_batch_ts        |count|
+-----------+----+------+---------+---------+-------------+-----------+------------+---------+------------+-----------+----------------+----------+-------------+---------------------+-----+
|15749093   |43.0|Male  |France   |0        |48           |2022-09-02 |801         |158713.08|2           |0          |98586.14        |0         |customers.csv|2026-09-02 02:28:03.0|2    |
|15800890   |45.0|Female|France   |1        |72           |2020-09-02 |554         |0.0      |2           |1          |181204.5        |0         |customers.csv|2026-09-02 02:28:03.0|2    |
|15736533   |37.0|Female|Germany  |0        |60   

In [9]:
check_cols = [
    c for c in df_customers.columns
    if c not in ["row_id", "source_file", "load_batch_ts"]
]

df_customers.groupBy(check_cols) \
    .count() \
    .filter(F.col("count") > 1) \
    .count()

150

In [10]:
df_customers = df_customers.dropDuplicates()

# Date Opened_Customer

In [11]:
df_customers.groupBy("date_opened") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

+------------+-----+
|date_opened |count|
+------------+-----+
|2024-09-01  |1027 |
|2025-09-01  |1025 |
|2019-09-03  |1014 |
|2018-09-03  |1010 |
|2021-09-02  |998  |
|2023-09-02  |996  |
|2017-09-03  |971  |
|2022-09-02  |970  |
|2020-09-02  |954  |
|2016-09-03  |480  |
|2026-09-01  |405  |
|2024/09/01  |8    |
|2022/09/02  |6    |
|09-02-2021  |6    |
|02/09/2022  |6    |
|Sep 03, 2018|5    |
|2019/09/03  |5    |
|02-09-2022  |5    |
|2025/09/01  |5    |
|09-03-2016  |5    |
|2020/09/02  |4    |
|Sep 03, 2017|4    |
|01-09-2024  |4    |
|09-02-2023  |4    |
|03/09/2018  |4    |
|01-09-2026  |4    |
|2021/09/02  |4    |
|03-09-2019  |4    |
|2023/09/02  |4    |
|03-09-2018  |4    |
|03/09/2016  |3    |
|Sep 02, 2020|3    |
|01/09/2024  |3    |
|Sep 01, 2024|3    |
|03-09-2017  |3    |
|09-01-2024  |3    |
|09-03-2017  |3    |
|Sep 01, 2025|3    |
|03/09/2017  |2    |
|02-09-2020  |2    |
|09-02-2020  |2    |
|Sep 03, 2016|2    |
|02/09/2023  |2    |
|09-02-2022  |2    |
|09-01-2026  

* 2026-06-13
* 04/03/2026
* 07-03-2026
* May 31, 2026
* 2026/06/13

In [12]:
df_customers.filter(
    F.col("date_opened").isNull()
).count()


[Stage 19:==============================================>       (173 + 7) / 200]



0

In [13]:
# normalize the dates
df_customers = df_customers.withColumn(
    "date_opened",
    F.coalesce(
        F.to_date("date_opened", "yyyy-MM-dd"),
        F.to_date("date_opened", "MMM dd, yyyy"),
        F.to_date("date_opened", "dd/MM/yyyy"),
        F.to_date("date_opened", "dd-MM-yyyy"),
        F.to_date("date_opened", "yyyy/MM/dd"),
        F.to_date("date_opened", "MM-dd-yyyy")
    )
)

In [14]:
df_customers.select("date_opened") \
    .show(100, truncate=False)

df_customers.printSchema()

+-----------+
|date_opened|
+-----------+
|2019-09-03 |
|2021-09-02 |
|2020-09-02 |
|2017-09-03 |
|2018-09-03 |
|2019-09-03 |
|2020-09-02 |
|2021-09-02 |
|2017-09-03 |
|2021-09-02 |
|2022-09-02 |
|2024-09-01 |
|2019-09-03 |
|2017-09-03 |
|2017-09-03 |
|2018-09-03 |
|2025-09-01 |
|2016-09-03 |
|2021-09-02 |
|2021-09-02 |
|2018-09-03 |
|2025-09-01 |
|2021-09-02 |
|2020-09-02 |
|2018-09-03 |
|2023-09-02 |
|2024-09-01 |
|2017-09-03 |
|2019-09-03 |
|2021-09-02 |
|2019-09-03 |
|2020-09-02 |
|2024-09-01 |
|2019-09-03 |
|2025-09-01 |
|2025-09-01 |
|2018-09-03 |
|2019-09-03 |
|2017-09-03 |
|2019-09-03 |
|2016-09-03 |
|2017-09-03 |
|2025-09-01 |
|2024-09-01 |
|2024-09-01 |
|2020-09-02 |
|2025-09-01 |
|2021-09-02 |
|2024-09-01 |
|2021-09-02 |
|2017-09-03 |
|2018-09-03 |
|2022-09-02 |
|2016-03-09 |
|2017-09-03 |
|2024-09-01 |
|2024-09-01 |
|2022-09-02 |
|2021-09-02 |
|2018-09-03 |
|2025-09-01 |
|2024-09-01 |
|2020-09-02 |
|2018-09-03 |
|2017-09-03 |
|2025-09-01 |
|2025-09-01 |
|2024-09-01 |
|2023-

## Spark parses different formats and DATE is Standardized

# Age

In [15]:
df_customers.groupBy("age") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

+--------+-----+
|age     |count|
+--------+-----+
|37.0    |468  |
|35.0    |467  |
|38.0    |467  |
|36.0    |446  |
|34.0    |440  |
|33.0    |433  |
|40.0    |425  |
|39.0    |416  |
|32.0    |413  |
|31.0    |398  |
|41.0    |357  |
|29.0    |341  |
|30.0    |322  |
|42.0    |312  |
|43.0    |292  |
|28.0    |268  |
|44.0    |253  |
|45.0    |223  |
|46.0    |222  |
|27.0    |206  |
|26.0    |197  |
|47.0    |171  |
|48.0    |166  |
|25.0    |148  |
|49.0    |146  |
|50.0    |132  |
|24.0    |131  |
|51.0    |119  |
|52.0    |100  |
|23.0    |98   |
|54.0    |83   |
|null    |82   |
|55.0    |82   |
|22.0    |81   |
|57.0    |74   |
|53.0    |72   |
|56.0    |68   |
|58.0    |67   |
|60.0    |61   |
|59.0    |60   |
|21.0    |53   |
|61.0    |52   |
|62.0    |52   |
|20.0    |40   |
|63.0    |39   |
|64.0    |37   |
|67.0    |37   |
|66.0    |34   |
|71.0    |27   |
|19.0    |26   |
|69.0    |22   |
|18.0    |22   |
|72.0    |20   |
|68.0    |19   |
|74.0    |18   |
|65.0    |17  

- Formats like `yeas x` and `x.0` are found
- inconsistent values like `-1` and `9999` are found

In [16]:
# NULL number
df_customers.filter(
    F.col("age").isNull()
).count()

82

In [17]:
df_customers = df_customers.withColumn(
    "age",
    F.regexp_extract(F.col("age"), r"(\d+)", 1).cast("int")
)

- "years 42"  → 42
- "25"        → 25
- String      → Integer

In [18]:
df_customers.select("age").show(50)
df_customers.printSchema()

+---+
|age|
+---+
| 41|
| 38|
| 32|
| 34|
| 32|
| 24|
| 37|
| 47|
| 34|
| 41|
| 44|
| 40|
| 37|
| 43|
| 35|
| 50|
| 45|
| 47|
| 22|
| 35|
| 48|
| 39|
| 53|
| 41|
| 23|
| 50|
| 52|
| 25|
| 46|
| 39|
| 36|
| 45|
| 40|
| 29|
| 37|
| 30|
| 35|
| 37|
| 25|
| 34|
| 48|
| 24|
| 61|
| 36|
| 36|
| 64|
| 34|
| 28|
| 29|
| 32|
+---+
only showing top 50 rows

root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- geography: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- tenure_months: string (nullable = true)
 |-- date_opened: date (nullable = true)
 |-- credit_score: string (nullable = true)
 |-- balance: string (nullable = true)
 |-- num_products: string (nullable = true)
 |-- has_cr_card: string (nullable = true)
 |-- estimated_salary: string (nullable = true)
 |-- is_churned: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_batch_ts: string (nullable = true)




[Stage 31:=========>                                                (1 + 5) / 6]



In [19]:
df_customers.filter(
    (F.col("age") < 15) | (F.col("age") > 100)
).select("age").show(50)

+----+
| age|
+----+
| 121|
|   1|
|   5|
|9999|
| 999|
| 150|
| 121|
|9999|
| 121|
|9999|
| 150|
| 999|
| 121|
|   1|
|   1|
|   1|
| 150|
| 999|
|   5|
|   1|
|9999|
| 999|
|   1|
|   5|
|9999|
|   5|
|   5|
|   1|
|   1|
|   1|
|   1|
|9999|
|   1|
|   1|
|9999|
| 121|
|   5|
|   5|
|   1|
|   5|
|   1|
|   1|
|   5|
| 999|
|   1|
|   1|
+----+



In [20]:
df_customers = df_customers.withColumn(
    "age",
    F.when(
        (F.col("age") < 15) | (F.col("age") > 100),
        None
    ).otherwise(F.col("age"))
)

In [21]:
df_customers.filter(
    F.col("age").isNull()
).count()

128

## Age is cleaned for type + domain validity, and invalid/missing values are NULL.

# Gender and Geograpghy 

In [22]:
df_customers.groupBy("geography") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)


[Stage 49:================================>                     (119 + 9) / 200]



+---------+-----+
|geography|count|
+---------+-----+
|France   |4940 |
|Germany  |2475 |
|Spain    |2437 |
|null     |80   |
|france   |31   |
|ES       |22   |
|GERMANY  |15   |
+---------+-----+



In [23]:
df_customers.groupBy("gender") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

+------+-----+
|gender|count|
+------+-----+
|Male  |5393 |
|Female|4488 |
|male  |45   |
|null  |37   |
|FEMALE|37   |
+------+-----+



In [24]:
# Normalize geography
df_customers = df_customers.withColumn(
    "geography",
    F.when(F.upper(F.col("geography")).isin("FRANCE", "FR"),"France")
    .when(F.upper(F.col("geography")).isin("GERMANY", "DE"),"Germany")
    .when(F.upper(F.col("geography")).isin("SPAIN", "ES"),"Spain")
    .otherwise(F.col("geography"))
)


# Normalize gender
df_customers = df_customers.withColumn(
    "gender",
    F.when(F.upper(F.col("gender")).isin("MALE", "M"),"Male")
    .when(F.upper(F.col("gender")).isin("FEMALE", "F"),"Female")
    .otherwise(F.col("gender"))
)

In [25]:
df_customers.groupBy("geography") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

df_customers.groupBy("gender") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

+---------+-----+
|geography|count|
+---------+-----+
|France   |4971 |
|Germany  |2490 |
|Spain    |2459 |
|null     |80   |
+---------+-----+

+------+-----+
|gender|count|
+------+-----+
|Male  |5438 |
|Female|4525 |
|null  |37   |
+------+-----+



## is_active and is_churned

In [26]:
df_customers.groupBy("is_active") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()
df_customers.groupBy("is_churned") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

+---------+-----+
|is_active|count|
+---------+-----+
|        1| 5151|
|        0| 4849|
+---------+-----+



+----------+-----+
|is_churned|count|
+----------+-----+
|         0| 7963|
|         1| 2037|
+----------+-----+



In [27]:
print(df_customers.filter(F.col("is_active").isNull()).count())
print(df_customers.filter(F.col("is_churned").isNull()).count())

0
0


In [28]:
df_customers = df_customers.withColumn("is_active",F.col("is_active").cast("boolean"))
df_customers = df_customers.withColumn("is_churned",F.col("is_churned").cast("boolean"))

In [29]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- geography: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- tenure_months: string (nullable = true)
 |-- date_opened: date (nullable = true)
 |-- credit_score: string (nullable = true)
 |-- balance: string (nullable = true)
 |-- num_products: string (nullable = true)
 |-- has_cr_card: string (nullable = true)
 |-- estimated_salary: string (nullable = true)
 |-- is_churned: boolean (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_batch_ts: string (nullable = true)



# num_products

In [30]:
df_customers.groupBy("num_products") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

+------------+-----+
|num_products|count|
+------------+-----+
|           1| 5049|
|           2| 4556|
|           3|  263|
|           4|   60|
|       three|   20|
|        many|   13|
|         two|   11|
|          -1|    9|
|           7|    7|
|           0|    6|
|          99|    6|
+------------+-----+



### `num_products` — EDA Findings

The column contains both valid numeric values and corrupted representations:

- `1`, `2`, `3` ,`4` → valid values From data description
- `"three"` → should be mapped to `3`
- `"two"` → should be mapped to `2`
- `"many"` → invalid/unknown → `NULL`
- `7`, `99` → suspicious values that need validation
- `0`, `-1` → invalid values

**Cleaning required:**
1. Convert `"three"` → `3`
2. Convert `"two"` → `2`
3. Convert `"many"` → `NULL`
4. Handle invalid/suspicious numeric values (`-1`, `0`, `7`, `99`)
5. Convert the column to `integer`

In [31]:
# mapping values
df_customers = df_customers.withColumn(
    "num_products",
    F.when(F.col("num_products") == "three", 3)
     .when(F.col("num_products") == "two", 2)
     .when(F.col("num_products") == "many", None)
     .otherwise(F.col("num_products").cast("int"))
)
df_customers.groupBy("num_products").count().orderBy("num_products").show()

+------------+-----+
|num_products|count|
+------------+-----+
|        null|   13|
|          -1|    9|
|           0|    6|
|           1| 5049|
|           2| 4567|
|           3|  283|
|           4|   60|
|           7|    7|
|          99|    6|
+------------+-----+



In [32]:
# fix invalid num_products
df_customers = df_customers.withColumn(
    "num_products",
    F.when(
        (F.col("num_products") < 1) | (F.col("num_products") > 4),
        None
    )
    .otherwise(F.col("num_products"))
)

In [33]:
df_customers.groupBy("num_products") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

+------------+-----+
|num_products|count|
+------------+-----+
|           1| 5049|
|           2| 4567|
|           3|  283|
|           4|   60|
|        null|   41|
+------------+-----+



# Credit Score

In [34]:
df_customers.groupBy("credit_score") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

+------------+-----+
|credit_score|count|
+------------+-----+
|         850|  230|
|         678|   63|
|         705|   53|
|         667|   53|
|         655|   53|
|         684|   51|
|         670|   50|
|         648|   48|
|         651|   48|
|         640|   47|
|         652|   47|
|         683|   47|
|         663|   47|
|         660|   47|
|         682|   47|
|         710|   45|
|         714|   45|
|         633|   45|
|         679|   45|
|         637|   45|
+------------+-----+
only showing top 20 rows



In [35]:
df_customers.filter(
    F.lower(F.col("credit_score")).isin(
        "six hundred",
        "score_700",
        "unknown"
    )
).groupBy("credit_score").count().show()

+------------+-----+
|credit_score|count|
+------------+-----+
|   score_700|   23|
| six hundred|   18|
+------------+-----+



### `credit_score` — EDA Findings

The column contains corrupted categorical representations:

- `"six hundred"` → should be mapped to `600`
- `"score_700"` → should be mapped to `700`
- `"unknown"` was mapped to `NULL` at first
 

In [36]:
df_customers = df_customers.withColumn(
    "credit_score",
    F.when(F.col("credit_score") == "six hundred", 600)
     .when(F.col("credit_score") == "score_700", 700)
     .otherwise(F.col("credit_score").cast("int"))
)
df_customers.groupBy("credit_score").count().show(20)

+------------+-----+
|credit_score|count|
+------------+-----+
|         471|    7|
|         496|   11|
|         833|    9|
|         463|    4|
|         540|   24|
|         623|   32|
|         737|   22|
|         516|   26|
|         808|    9|
|         580|   30|
|         451|    5|
|         458|    7|
|         588|   34|
|         804|    8|
|         799|   12|
|         481|   10|
|         472|    8|
|         513|   17|
|         633|   45|
|         673|   35|
+------------+-----+
only showing top 20 rows



In [37]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- geography: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- tenure_months: string (nullable = true)
 |-- date_opened: date (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- balance: string (nullable = true)
 |-- num_products: integer (nullable = true)
 |-- has_cr_card: string (nullable = true)
 |-- estimated_salary: string (nullable = true)
 |-- is_churned: boolean (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_batch_ts: string (nullable = true)



# balance

In [38]:
df_customers.groupBy("balance") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20)

+---------+-----+
|  balance|count|
+---------+-----+
|      0.0| 3575|
|     null|   62|
|9999999.0|   14|
|       -1|   13|
|  -5000.0|   10|
|     9999|    9|
|     -1.0|    8|
|105473.74|    2|
|130170.82|    2|
|148163.57|    1|
|141005.47|    1|
| 105239.1|    1|
|133707.09|    1|
|110463.25|    1|
| 62052.28|    1|
| 59893.85|    1|
|170008.84|    1|
|109869.32|    1|
| 80615.46|    1|
|150358.97|    1|
+---------+-----+
only showing top 20 rows



### `balance` — EDA Findings

The column contains:

- `0.0` → valid value
- `NULL` → missing values
- `9999999.0` → suspicious extreme outlier
- `-1`, `-1.0` → invalid negative sentinel values
- `-5000.0` → suspicious/invalid negative value
- `9999` → suspicious value that needs validation
- Other decimal values → normal balance values

**Cleaning required:**
1. Convert `balance` from string to `double`
2. Handle negative values (`-1`, `-5000`)
3. Investigate extreme values such as `9999999`
4. Investigate `9999`
5. Handle remaining NULL values separately

In [39]:
df_customers.filter(
    F.col("balance").isin("-1", "-1.0", "-5000.0", "9999", "9999999.0")
).groupBy("balance").count().show()

+---------+-----+
|  balance|count|
+---------+-----+
|       -1|   13|
|9999999.0|   14|
|     9999|    9|
|     -1.0|    8|
|  -5000.0|   10|
+---------+-----+



In [40]:
df_customers = df_customers.withColumn(
    "balance",
    F.when(
        F.col("balance").isin("-1", "-1.0", "-5000.0", "9999", "9999999.0"),
        None
    )
    .otherwise(F.col("balance"))
)

In [41]:
df_customers.select(
    F.min("balance").alias("min"),
    F.max("balance").alias("max"),
    F.avg("balance").alias("avg")
).show()


[Stage 127:===============================>                     (117 + 6) / 200]



+---+--------+-----------------+
|min|     max|              avg|
+---+--------+-----------------+
|0.0|99986.98|76472.81348644278|
+---+--------+-----------------+



In [42]:
df_customers = df_customers.withColumn(
    "balance",
    F.col("balance").cast("double")
)

In [43]:
df_customers.groupBy("balance").count().orderBy(F.desc("count")).show(20)

+---------+-----+
|  balance|count|
+---------+-----+
|      0.0| 3575|
|     null|  116|
|105473.74|    2|
|130170.82|    2|
| 130231.8|    1|
| 86616.35|    1|
|110240.04|    1|
|182491.57|    1|
|109330.06|    1|
|170331.37|    1|
|144961.97|    1|
| 99844.68|    1|
|128736.39|    1|
|119684.88|    1|
| 77556.79|    1|
|127009.83|    1|
|112949.71|    1|
| 125734.2|    1|
| 89763.84|    1|
|145018.64|    1|
+---------+-----+
only showing top 20 rows



# has_cr_card

In [44]:
df_customers.groupBy("has_cr_card") \
    .count() \
    .orderBy("has_cr_card") \
    .show()

+-----------+-----+
|has_cr_card|count|
+-----------+-----+
|          0| 2945|
|          1| 7055|
+-----------+-----+



In [45]:
df_customers = df_customers.withColumn("has_cr_card",F.col("has_cr_card").cast("boolean"))

# load_batch_ts

In [46]:
df_customers.groupBy("load_batch_ts") \
    .count() \
    .orderBy("load_batch_ts") \
    .show()

+--------------------+-----+
|       load_batch_ts|count|
+--------------------+-----+
|2026-09-02 02:28:...|10000|
+--------------------+-----+



### `load_batch_ts` — EDA Findings

- One timestamp value is present across all records.
- No obvious corruption was found.
- Convert from `string` to Spark `timestamp`.

In [47]:
df_customers = df_customers.withColumn(
    "load_batch_ts",
    F.to_timestamp("load_batch_ts")
)

In [48]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- geography: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- tenure_months: string (nullable = true)
 |-- date_opened: date (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- balance: double (nullable = true)
 |-- num_products: integer (nullable = true)
 |-- has_cr_card: boolean (nullable = true)
 |-- estimated_salary: string (nullable = true)
 |-- is_churned: boolean (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_batch_ts: timestamp (nullable = true)



# estimated_salary

In [49]:
df_customers.groupBy("estimated_salary") \
    .count() \
    .orderBy("estimated_salary") \
    .show()

+----------------+-----+
|estimated_salary|count|
+----------------+-----+
|       100015.79|    1|
|       100060.54|    1|
|        100075.1|    1|
|        10008.68|    1|
|       100101.06|    1|
|       100127.71|    1|
|       100130.95|    1|
|        100137.7|    1|
|        10014.72|    1|
|       100153.43|    1|
|       100183.05|    1|
|       100187.43|    1|
|         1002.39|    1|
|        100200.4|    1|
|        10023.15|    1|
|       100236.02|    1|
|        100240.2|    1|
|       100304.13|    1|
|       100324.01|    1|
|       100335.55|    1|
+----------------+-----+
only showing top 20 rows



In [50]:
df_customers.filter(
    F.col("estimated_salary").isNull()
).count()

0

In [51]:
df_customers = df_customers.withColumn(
    "estimated_salary",
    F.col("estimated_salary").cast("double")
)

In [52]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- geography: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- tenure_months: string (nullable = true)
 |-- date_opened: date (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- balance: double (nullable = true)
 |-- num_products: integer (nullable = true)
 |-- has_cr_card: boolean (nullable = true)
 |-- estimated_salary: double (nullable = true)
 |-- is_churned: boolean (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_batch_ts: timestamp (nullable = true)



# tenure_months

In [96]:
df_customers.groupBy("tenure_months") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

+-------------+-----+
|tenure_months|count|
+-------------+-----+
|24           |1048 |
|12           |1032 |
|84           |1023 |
|96           |1021 |
|60           |1009 |
|36           |1005 |
|48           |986  |
|108          |980  |
|72           |962  |
|120          |488  |
|0            |412  |
|999          |13   |
|-1           |11   |
|-10          |10   |
+-------------+-----+



In [98]:
df_customers = df_customers.withColumn(
    "tenure_months",
    F.when(
        (F.col("tenure_months") < 0) |
        (F.col("tenure_months") == 999),
        None
    ).otherwise(F.col("tenure_months").cast("int"))
)

### `tenure_months` — EDA Findings

The `tenure_months` column was stored as a `STRING`.

The column contains valid tenure values such as `0, 12, 24, 36, 48, 60, 72, 84, 96, 108, 120` months.

Some invalid values were also found:
- `999` → invalid/corrupted value
- `-1` → invalid negative value
- `-10` → invalid negative value

**Cleaning applied:**
- Converted `tenure_months` from `STRING` to `INTEGER`.
- Replaced `999` and negative values with `NULL`.
- Preserved all valid tenure values.

### Customer Features — Data Cleaning Summary

The `Customer_Features` table was cleaned and standardized:

- `age` → validated and cleaned
- `date_opened` → standardized to `DATE`
- `gender` → standardized to `Male` / `Female`
- `geography` → standardized to `France` / `Germany` / `Spain`
- `is_active` → converted to `BOOLEAN`
- `is_churned` → converted to `BOOLEAN`
- `tenure_months` → converted and validated
- `credit_score` → converted to `INTEGER` and cleaned
- `balance` → converted to `DOUBLE` and invalid values handled
- `num_products` → converted to `INTEGER` and invalid values handled
- `has_cr_card` → converted to `BOOLEAN`
- `estimated_salary` → converted to `DOUBLE`
- `load_batch_ts` → converted to `TIMESTAMP`
- `row_id` → removed as unnecessary technical metadata
- Duplicate customer records → removed

The resulting schema contains appropriate data types and cleaned customer features.

In [53]:
# Check NULLs
df_customers.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_customers.columns
]).show()

+-----------+---+------+---------+---------+-------------+-----------+------------+-------+------------+-----------+----------------+----------+-----------+-------------+
|customer_id|age|gender|geography|is_active|tenure_months|date_opened|credit_score|balance|num_products|has_cr_card|estimated_salary|is_churned|source_file|load_batch_ts|
+-----------+---+------+---------+---------+-------------+-----------+------------+-------+------------+-----------+----------------+----------+-----------+-------------+
|          0|128|    37|       80|        0|            0|          0|          17|    116|          41|          0|               0|         0|          0|            0|
+-----------+---+------+---------+---------+-------------+-----------+------------+-------+------------+-----------+----------------+----------+-----------+-------------+



## impute N/A values  

In [91]:
# Fill categorical NULLs with the mode

for c in ["gender", "geography"]:
    mode_value = (
        df_customers
        .filter(F.col(c).isNotNull())
        .groupBy(c)
        .count()
        .orderBy(F.desc("count"))
        .first()[0]
    )

    df_customers = df_customers.fillna({c: mode_value})

    print(f"{c} mode: {mode_value}")

gender mode: Male


geography mode: France


# IQR for Detecting Outiers

In [92]:
def IQR(df, column):
    q1, q3 = df.approxQuantile(column, [0.25, 0.75], 0.01)
    median = df.approxQuantile(column, [0.5], 0.01)[0]

    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    print(f"{column}")
    print(f"Q1     : {q1}")
    print(f"Q3     : {q3}")
    print(f"Median : {median}")
    print(f"IQR    : {iqr}")
    print(f"Lower  : {lower}")
    print(f"Upper  : {upper}")

    return median

In [93]:
age_median = IQR(df_customers, "age")

age
Q1     : 32.0
Q3     : 44.0
Median : 37.0
IQR    : 12.0
Lower  : 14.0
Upper  : 62.0


`age NULL` ===>> `Median`  

In [94]:
df_customers = df_customers.fillna({
    "age": age_median
})

In [99]:
for c in ["credit_score", "balance", "num_products", "tenure_months", "estimated_salary"]:
    print("\n" + "=" * 40)
    median = IQR(df_customers, c)


credit_score
Q1     : 582.0
Q3     : 715.0
Median : 650.0
IQR    : 133.0
Lower  : 382.5
Upper  : 914.5

balance
Q1     : 0.0
Q3     : 127146.68
Median : 96402.96
IQR    : 127146.68
Lower  : -190720.02
Upper  : 317866.69999999995

num_products
Q1     : 1.0
Q3     : 2.0
Median : 1.0
IQR    : 1.0
Lower  : -0.5
Upper  : 3.5



tenure_months
Q1     : 24.0
Q3     : 84.0
Median : 60.0
IQR    : 60.0
Lower  : -66.0
Upper  : 174.0



estimated_salary
Q1     : 49707.85
Q3     : 147963.07
Median : 98453.45
IQR    : 98255.22
Lower  : -97674.98000000001
Upper  : 295345.9


In [100]:
credit_score_median = IQR(df_customers, "credit_score")

df_customers = df_customers.fillna({
    "credit_score": credit_score_median
})

credit_score
Q1     : 582.0
Q3     : 715.0
Median : 650.0
IQR    : 133.0
Lower  : 382.5
Upper  : 914.5


In [101]:
balance_median = IQR(df_customers, "balance")

df_customers = df_customers.fillna({
    "balance": balance_median
})

balance
Q1     : 0.0
Q3     : 127146.68
Median : 96462.25
IQR    : 127146.68
Lower  : -190720.02
Upper  : 317866.69999999995


In [102]:
num_products_mode = (
    df_customers
    .filter(F.col("num_products").isNotNull())
    .groupBy("num_products")
    .count()
    .orderBy(F.desc("count"))
    .first()["num_products"]
)

print("num_products mode:", num_products_mode)

df_customers = df_customers.fillna({
    "num_products": num_products_mode
})


[Stage 287:==================================>                  (129 + 7) / 200]



num_products mode: 1


### Missing Values — Imputation

The following missing values were handled using simple imputation methods based on the data type and characteristics of each feature:

- `gender` → Filled missing values with the **mode**.
- `geography` → Filled missing values with the **mode**.
- `age` → Filled missing values with the **median** (`37`).
- `credit_score` → Filled missing values with the **median** (`650`).
- `balance` → Filled missing values with the **median** (`96,402.96`).
- `num_products` → Filled missing values with the **mode**.

These steps were performed to handle missing values while preserving the existing valid data.

# Offers PreProcessing

In [54]:
df_offers = spark.read.option("header", True).csv(f"{BASE}/offers")

print("Raw rows:", df_offers.count())
df_offers.printSchema()

Raw rows: 8127
root
 |-- offer_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- offer_type: string (nullable = true)
 |-- accepted: string (nullable = true)
 |-- date_offered: string (nullable = true)



In [55]:
# Trim whitespace from all columns
cols = df_offers.columns

for c in cols:
    df_offers = df_offers.withColumn(c,F.trim(F.col(c)))

# offer_type

In [56]:
df_offers.groupBy("offer_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

+---------------+-----+
|offer_type     |count|
+---------------+-----+
|Fee Waiver     |1612 |
|Cashback       |1605 |
|Loyalty Bonus  |1587 |
|Product Upgrade|1584 |
|Rate Discount  |1554 |
|null           |35   |
|RATE DISCOUNT  |31   |
|fee waiver     |28   |
|cashback       |25   |
|product upgrade|23   |
|N/A            |16   |
|Unknown        |16   |
|NULL           |11   |
+---------------+-----+



### `offer_type` — EDA Findings

The column contains the expected offer types, but some values have inconsistent casing and missing/sentinel representations:

- `Fee Waiver`, `Cashback`, `Loyalty Bonus`, `Product Upgrade`, `Rate Discount` → valid values
- Lowercase values such as `fee waiver`, `cashback`, `product upgrade` → inconsistent casing
- `RATE DISCOUNT` → inconsistent casing
- `Unknown`, `N/A`, `NULL`, and actual `NULL` → missing/unknown values

**Cleaning required:**
1. Standardize the casing of offer types.
2. Convert `Unknown`, `N/A`, and `NULL` strings to actual `NULL`.
3. Preserve actual `NULL` values.

In [57]:
# Normalize offer_type
df_offers = df_offers.withColumn(
    "offer_type",
    F.when(F.lower("offer_type") == "product upgrade", "Product Upgrade")
     .when(F.lower("offer_type") == "rate discount", "Rate Discount")
     .when(F.lower("offer_type") == "cashback", "Cashback")
     .when(F.lower("offer_type") == "fee waiver", "Fee Waiver")
     .otherwise(F.col("offer_type"))
)

In [58]:
# Convert sentinel values to NULL
for c in ["offer_type", "accepted", "date_offered"]:
    df_offers = df_offers.withColumn(
        c,
        F.when(
            F.upper(F.trim(F.col(c))).isin("N/A", "UNKNOWN", "NULL"),
            None
        ).otherwise(F.col(c))
    )

### `offer_type` — Transformation

- `fee waiver` → `Fee Waiver`
- `cashback` → `Cashback`
- `product upgrade` → `Product Upgrade`
- `RATE DISCOUNT` → `Rate Discount`
- `Unknown`, `N/A`, `NULL`→ `NULL`
- Actual `NULL` → preserved as `NULL`

The `offer_type` values are now standardized.

In [59]:
df_offers.groupBy("offer_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

+---------------+-----+
|     offer_type|count|
+---------------+-----+
|     Fee Waiver| 1640|
|       Cashback| 1630|
|Product Upgrade| 1607|
|  Loyalty Bonus| 1587|
|  Rate Discount| 1585|
|           null|   78|
+---------------+-----+



# accepted

In [60]:
df_offers.groupBy("accepted") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

+--------+-----+
|accepted|count|
+--------+-----+
|False   |3990 |
|True    |3925 |
|null    |99   |
|MAYBE   |32   |
|TRUEE   |26   |
|2       |21   |
|1       |9    |
|TRUE    |8    |
|yes     |7    |
|0       |6    |
|no      |4    |
+--------+-----+



### `accepted` — EDA Findings

- `True`, `TRUE`, `1`, `yes` → `True`
- `False`, `0`, `no` → `False`
- `TRUEE` → corrupted representation of `True`
- `MAYBE`, `2` → invalid/unknown → `NULL`
- Actual `NULL` → preserved

In [61]:
df_offers = df_offers.withColumn(
    "accepted",
    F.when(F.upper("accepted").isin("TRUE", "TRUEE", "1", "YES"), True)
     .when(F.upper("accepted").isin("FALSE", "0", "NO"), False)
     .otherwise(None)
)

In [62]:
df_offers.printSchema()

root
 |-- offer_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- offer_type: string (nullable = true)
 |-- accepted: boolean (nullable = true)
 |-- date_offered: string (nullable = true)



# Date offered

In [63]:
df_offers.select("date_offered") \
    .distinct() \
    .show(100, truncate=False)

+------------+
|date_offered|
+------------+
|2026-02-01  |
|2024-10-24  |
|07-31-2026  |
|2024-09-15  |
|06-12-2025  |
|2025-02-25  |
|2025-06-03  |
|2024-10-22  |
|2026/03/02  |
|04-19-2026  |
|2025-08-17  |
|2026-02-27  |
|2024-12-12  |
|2025-04-05  |
|2025-03-20  |
|2026-07-04  |
|2026-07-23  |
|2025-09-14  |
|2026-03-15  |
|07-22-2025  |
|2025-12-07  |
|2026-04-15  |
|2026-06-05  |
|2025-01-03  |
|21/07/2025  |
|2025-08-18  |
|2025-12-30  |
|Apr 07, 2025|
|2024-10-10  |
|2024-12-09  |
|2025-06-16  |
|2024-11-20  |
|2024-09-21  |
|2025-07-15  |
|2024-12-03  |
|Apr 09, 2025|
|2025-09-30  |
|2024-11-08  |
|2026-01-30  |
|2026-04-07  |
|2026-08-24  |
|Jun 02, 2025|
|15-12-2025  |
|2024-11-30  |
|2025-05-06  |
|2025-07-21  |
|2025-05-23  |
|2025-04-12  |
|2026-02-15  |
|2025-10-09  |
|2024-11-24  |
|2026-01-05  |
|2026-02-05  |
|2025-01-23  |
|2026-01-28  |
|2024-09-10  |
|2025-07-16  |
|2025-06-14  |
|2026-02-25  |
|2025-01-25  |
|2025-01-28  |
|06-27-2026  |
|2024-09-22  |
|2025-03-2

### `date_offered` — EDA Findings

The column contains dates in multiple formats:

- `yyyy-MM-dd` → `2026-02-01`
- `MM-dd-yyyy` → `07-31-2026`
- `yyyy/MM/dd` → `2026/03/02`
- `dd/MM/yyyy` → `21/07/2025`
- `dd-MM-yyyy` → `15-12-2025`
- `MMM dd, yyyy` → `Apr 07, 2025`

This shows that the date format was corrupted/inconsistent.

**Cleaning required:**
- Parse all supported date formats.
- Standardize the column to Spark `DATE`.

In [64]:
# Parse dates
df_offers = df_offers.withColumn(
    "date_offered",
    F.coalesce(
        F.to_date("date_offered", "yyyy-MM-dd"),
        F.to_date("date_offered", "MMM dd, yyyy"),
        F.to_date("date_offered", "dd/MM/yyyy"),
        F.to_date("date_offered", "dd-MM-yyyy"),
        F.to_date("date_offered", "yyyy/MM/dd"),
        F.to_date("date_offered", "MM-dd-yyyy")
    )
)

In [65]:
df_offers.printSchema()

root
 |-- offer_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- offer_type: string (nullable = true)
 |-- accepted: boolean (nullable = true)
 |-- date_offered: date (nullable = true)



# Check duplicates

In [66]:
duplicate_offers = df_offers.groupBy("offer_id") \
    .count() \
    .filter(
        F.col("offer_id").isNotNull() & (F.col("count") > 1)
    )

print("Duplicate offer_id groups:", duplicate_offers.count())

Duplicate offer_id groups: 107


In [67]:
df_offers = df_offers.dropDuplicates(["offer_id"])

In [68]:
valid_customers = df_customers.select("customer_id").distinct()

orphan_offers = (
    df_offers
    .filter(F.col("customer_id").isNotNull())
    .join(valid_customers, "customer_id", "left_anti")
)

print("Orphan customer_id rows:", orphan_offers.count())

Orphan customer_id rows: 144


In [69]:
df_offers = df_offers.join(
    valid_customers,
    "customer_id",
    "left_semi"
)

### Offers — Final Validation

- 107 duplicate `offer_id` groups were identified and removed.
- NULL `offer_id` values were preserved.
- 144 orphan `customer_id` rows were identified.
- The 144 orphan records were removed because their `customer_id` does not exist in `Customer_Features`.
- `customer_id` is now restricted to valid customer records.

In [70]:
# Check NULLs
df_offers.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_offers.columns
]).show()

+-----------+--------+----------+--------+------------+
|customer_id|offer_id|offer_type|accepted|date_offered|
+-----------+--------+----------+--------+------------+
|          0|       0|        78|     152|          64|
+-----------+--------+----------+--------+------------+




[Stage 175:===========================================>         (163 + 6) / 200]



# Dealing with N/A

In [108]:
# Impute missing offer_type with "Unknown"

df_offers = df_offers.withColumn(
    "offer_type",
    F.coalesce(F.col("offer_type"), F.lit("Unknown"))
)

setting all `N/A` to `Unknown` not to ruin the data or making it bias 

In [109]:
# Preserve accepted as nullable boolean, add explicit tri-state string for warehouse loading

df_offers = df_offers.withColumn(
    "accepted_flag",
    F.when(F.col("accepted") == True, "Yes")
     .when(F.col("accepted") == False, "No")
     .otherwise("Unknown")
)

In [110]:
# Impute missing date_offered with sentinel date for warehouse date-dimension join

df_offers = df_offers.withColumn(
    "date_offered_key",
    F.coalesce(F.col("date_offered"), F.to_date(F.lit("1900-01-01")))
)

### Offers — Missing Value Imputation

Before loading into the warehouse, missing values in `offer_type`, `accepted`, and `date_offered` were handled as follows. Rows were **not dropped** for missing dimensional attributes — only `offer_id` nulls would justify dropping a row, since that breaks the fact table's grain.

**`offer_type` (78 nulls)**
- Imputed with `"Unknown"` — a low missing rate on a categorical attribute, safe to fill with an explicit placeholder rather than lose the row.

**`accepted` (152 nulls)**
- Left as a nullable boolean (no True/False was guessed, to avoid fabricating signal in the acceptance rate).
- Added `accepted_flag` (`"Yes"` / `"No"` / `"Unknown"`) as an explicit tri-state column for warehouse loading, so NULLs are never silently coerced to False downstream.

**`date_offered` (64 nulls)**
- No statistical imputation (mean/mode) was used, since there's no meaningful "average" offer date and a fabricated date would corrupt recency-based features.
- Added `date_offered_key`, coalescing nulls to a sentinel date (`1900-01-01`), so the row can still join cleanly to a `Dim_Date` table without breaking on NULL foreign keys.
- The original `date_offered` column is preserved (still nullable) for feature engineering steps (e.g. "days since last offer"), where sentinel/null rows should be filtered out explicitly rather than treated as real dates.

# Customer support tickets PreProcessing

In [71]:
df_tickets = spark.read.option("header", True).csv(f"{BASE}/customer_support_tickets")

print("Raw rows:", df_tickets.count())
df_tickets.printSchema()

Raw rows: 14182
root
 |-- ticket_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- resolution_time_hrs: string (nullable = true)



In [72]:
# trim white spaces
for c in df_tickets.columns:
    df_tickets = df_tickets.withColumn(c, F.trim(F.col(c)))

# Issue Type

In [73]:
df_tickets.groupBy("issue_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

+------------+-----+
|issue_type  |count|
+------------+-----+
|App Bug     |2364 |
|Loan Query  |2351 |
|Billing     |2347 |
|Fraud Alert |2305 |
|Card Problem|2291 |
|Login Issue |2267 |
|null        |79   |
|NULL        |34   |
|login issue |26   |
|Unknown     |23   |
|app bug     |19   |
|loan query  |18   |
|N/A         |16   |
|billing     |15   |
|FRAUD ALERT |14   |
|CARD PROBLEM|13   |
+------------+-----+



In [74]:
# Normalize issue_type
df_tickets = df_tickets.withColumn(
    "issue_type",
    F.when(F.upper(F.col("issue_type")).isin("NULL", "UNKNOWN", "N/A"), None)
     .when(F.lower("issue_type") == "app bug", "App Bug")
     .when(F.lower("issue_type") == "loan query", "Loan Query")
     .when(F.lower("issue_type") == "billing", "Billing")
     .when(F.lower("issue_type") == "fraud alert", "Fraud Alert")
     .when(F.lower("issue_type") == "card problem", "Card Problem")
     .when(F.lower("issue_type") == "login issue", "Login Issue")
     .otherwise(F.col("issue_type"))
)

### `issue_type` — EDA Findings

The column contains the expected issue types, but some values have inconsistent casing and missing/sentinel representations:

- `App Bug`, `Loan Query`, `Billing`, `Fraud Alert`, `Card Problem`, `Login Issue` → valid values
- Lowercase and uppercase variants → inconsistent casing
- `Unknown`, `N/A`, `NULL`, and actual `NULL` → missing/unknown values

**Cleaning applied:**
- Standardized the casing of valid issue types.
- Converted `Unknown`, `N/A`, and `NULL` strings to actual `NULL`.
- Preserved actual `NULL` values.

In [75]:
df_tickets.groupBy("issue_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

+------------+-----+
|issue_type  |count|
+------------+-----+
|App Bug     |2383 |
|Loan Query  |2369 |
|Billing     |2362 |
|Fraud Alert |2319 |
|Card Problem|2304 |
|Login Issue |2293 |
|null        |152  |
+------------+-----+



# Severity

In [76]:
df_tickets.groupBy("severity") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

+--------+-----+
|severity|count|
+--------+-----+
|Low     |5500 |
|Medium  |4985 |
|High    |2531 |
|Critical|932  |
|null    |66   |
|MEDIUM  |42   |
|low     |40   |
|NULL    |26   |
|high    |20   |
|N/A     |19   |
|Unknown |18   |
|CRITICAL|3    |
+--------+-----+



In [77]:
df_tickets = df_tickets.withColumn(
    "severity",
    F.when(F.upper(F.col("severity")).isin("NULL", "N/A", "UNKNOWN"), None)
     .when(F.lower("severity") == "low", "Low")
     .when(F.lower("severity") == "medium", "Medium")
     .when(F.lower("severity") == "high", "High")
     .when(F.lower("severity") == "critical", "Critical")
     .otherwise(F.col("severity"))
)

In [78]:
df_tickets.groupBy("severity") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

+--------+-----+
|severity|count|
+--------+-----+
|Low     |5540 |
|Medium  |5027 |
|High    |2551 |
|Critical|935  |
|null    |129  |
+--------+-----+



### `severity` — EDA Findings

The column contains the expected severity levels, but some values have inconsistent casing and missing/sentinel representations:

- `Low`, `Medium`, `High`, `Critical` → valid values
- `MEDIUM`, `low`, `high`, `CRITICAL` → inconsistent casing
- `Unknown`, `N/A`, `NULL`, and actual `NULL` → missing/unknown values

**Cleaning applied:**
- Standardized the casing of severity levels.
- Converted `Unknown`, `N/A`, and `NULL` strings to actual `NULL`.
- Preserved actual `NULL` values.

In [79]:
df_tickets.printSchema()

root
 |-- ticket_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- resolution_time_hrs: string (nullable = true)



# Resolution Time

In [80]:
df_tickets.groupBy("resolution_time_hrs") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

+-------------------+-----+
|resolution_time_hrs|count|
+-------------------+-----+
|1.2                |250  |
|2.9                |245  |
|2.4                |243  |
|2.1                |239  |
|1.5                |238  |
|2.7                |236  |
|1.8                |236  |
|1.6                |233  |
|1.1                |233  |
|2.3                |230  |
|2.0                |228  |
|2.6                |228  |
|1.9                |228  |
|1.7                |221  |
|2.5                |220  |
|2.8                |218  |
|2.2                |217  |
|1.3                |210  |
|1.4                |209  |
|1.0                |132  |
|3.0                |125  |
|4.7                |100  |
|4.2                |99   |
|4.9                |95   |
|4.6                |94   |
|4.3                |91   |
|4.8                |85   |
|5.0                |82   |
|4.4                |81   |
|4.1                |79   |
|4.5                |79   |
|11.6               |70   |
|11.4               

In [81]:
df_tickets = df_tickets.withColumn(
    "resolution_time_hrs",
    F.col("resolution_time_hrs").cast("double")
)

In [82]:
df_tickets = df_tickets.withColumn(
    "resolution_time_hrs",
    F.when(
        (F.col("resolution_time_hrs") < 0) |
        F.col("resolution_time_hrs").isin(999, 9999, 2500),
        None
    ).otherwise(F.col("resolution_time_hrs"))
)

In [83]:
df_tickets.select(
    F.min("resolution_time_hrs").alias("min"),
    F.max("resolution_time_hrs").alias("max"),
    F.sum(F.col("resolution_time_hrs").isNull().cast("int")).alias("nulls")
).show()

+---+-----+-----+
|min|  max|nulls|
+---+-----+-----+
|1.0|119.9|  449|
+---+-----+-----+



### `resolution_time_hrs` — EDA Findings

The column contains resolution times represented as decimal hours. Several clearly invalid values were found:

- Negative values such as `-1.0` and `-5.0` → invalid
- `999.0`, `9999.0`, and `2500.0` → invalid/sentinel values
- Actual `NULL` values → missing data
- Other positive values, including values above 20 hours, were preserved as they can represent valid resolution times.

**Cleaning applied:**
- Converted `resolution_time_hrs` from `STRING` to `DOUBLE`.
- Converted negative values and the invalid sentinel values to actual `NULL`.
- Preserved valid positive resolution times.

In [84]:
# Check nulls
df_tickets.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_tickets.columns
]).show()

+---------+-----------+----------+--------+-------------------+
|ticket_id|customer_id|issue_type|severity|resolution_time_hrs|
+---------+-----------+----------+--------+-------------------+
|        0|          0|       152|     129|                449|
+---------+-----------+----------+--------+-------------------+



# Check duplicates 

In [85]:
cols = df_tickets.columns

df_tickets.groupBy(cols) \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

+---------+-----------+------------+--------+-------------------+-----+
|ticket_id|customer_id|issue_type  |severity|resolution_time_hrs|count|
+---------+-----------+------------+--------+-------------------+-----+
|7643     |15566253   |Billing     |Medium  |12.1               |2    |
|661      |15613786   |Card Problem|Critical|63.6               |2    |
|1716     |15650098   |Login Issue |Low     |1.0                |2    |
|4357     |15579781   |Login Issue |High    |14.8               |2    |
|4527     |15787550   |Login Issue |Low     |1.2                |2    |
|4904     |15754919   |Login Issue |Critical|68.2               |2    |
|4153     |15631159   |Login Issue |Low     |2.7                |2    |
|2782     |15780954   |App Bug     |Medium  |11.4               |2    |
|10555    |15792107   |Billing     |Low     |1.9                |2    |
|10736    |15633608   |Billing     |Medium  |6.4                |2    |
|13773    |15664035   |Login Issue |Low     |2.4                

In [86]:
duplicate_groups = df_tickets.groupBy(cols) \
    .count() \
    .filter(F.col("count") > 1)

print("Duplicate groups:", duplicate_groups.count())

Duplicate groups: 210


In [87]:
df_tickets = df_tickets.dropDuplicates(cols)

In [88]:
valid_customers = df_customers.select("customer_id").distinct()

orphan_tickets = (
    df_tickets
    .filter(F.col("customer_id").isNotNull())
    .join(
        valid_customers,
        "customer_id",
        "left_anti"
    )
)

print("Non-null orphan customer IDs:", orphan_tickets.count())

Non-null orphan customer IDs: 209


In [89]:
df_tickets = df_tickets.join(
    valid_customers,
    "customer_id",
    "left_semi"
)

In [90]:
df_tickets.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_tickets.columns
]).show()

+-----------+---------+----------+--------+-------------------+
|customer_id|ticket_id|issue_type|severity|resolution_time_hrs|
+-----------+---------+----------+--------+-------------------+
|          0|        0|       152|     129|                449|
+-----------+---------+----------+--------+-------------------+



### Duplicates & Orphan IDs — Final Validation

**Duplicate records:**
- Checked for exact duplicate records using all columns.
- 210 duplicate groups were identified.
- Duplicate records were removed.

**Orphan customer IDs:**
- Compared ticket `customer_id` values with valid `customer_id` values from `Customer_Features`.
- 209 non-null orphan records were identified.
- Orphan records were removed.

**Final NULL check:**
- `customer_id` → 0 NULLs
- `ticket_id` → 0 NULLs
- `issue_type` → 152 NULLs
- `severity` → 129 NULLs
- `resolution_time_hrs` → 449 NULLs

# N/A imputation

In [103]:
issue_type_mode = (
    df_tickets
    .filter(F.col("issue_type").isNotNull())
    .groupBy("issue_type")
    .count()
    .orderBy(F.desc("count"))
    .first()["issue_type"]
)

print("issue_type mode:", issue_type_mode)

df_tickets = df_tickets.fillna({
    "issue_type": issue_type_mode
})

issue_type mode: Loan Query


In [104]:
severity_mode = (
    df_tickets
    .filter(F.col("severity").isNotNull())
    .groupBy("severity")
    .count()
    .orderBy(F.desc("count"))
    .first()["severity"]
)

print("severity mode:", severity_mode)

df_tickets = df_tickets.fillna({
    "severity": severity_mode
})

severity mode: Low


In [105]:
resolution_median = IQR(df_tickets, "resolution_time_hrs")

resolution_time_hrs
Q1     : 2.4
Q3     : 15.3
Median : 6.5
IQR    : 12.9
Lower  : -16.950000000000003
Upper  : 34.650000000000006


In [106]:
df_tickets = df_tickets.fillna({
    "resolution_time_hrs": resolution_median
})

In [107]:
df_tickets.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_tickets.columns
]).show()

+-----------+---------+----------+--------+-------------------+
|customer_id|ticket_id|issue_type|severity|resolution_time_hrs|
+-----------+---------+----------+--------+-------------------+
|          0|        0|         0|       0|                  0|
+-----------+---------+----------+--------+-------------------+



### Missing Values — Customer Support Tickets

The remaining missing values in the customer support tickets dataset were handled based on the data type of each feature.

- `issue_type` → Filled missing values with the **mode**, which was `Loan Query`.
- `severity` → Filled missing values with the **mode**, which was `Low`.
- `resolution_time_hrs` → Filled missing values with the **median**, which was `6.5` hours.

For `resolution_time_hrs`, the distribution was examined using the IQR method:

- Q1 = `2.4`
- Q3 = `15.3`
- Median = `6.5`
- IQR = `12.9`

**Imputation applied:**
- Missing `issue_type` values → `Loan Query`
- Missing `severity` values → `Low`
- Missing `resolution_time_hrs` values → `6.5`

# Usage 

In [118]:
df_usage = (
    spark.read
    .option("multiLine", True)
    .json(f"{BASE}/usage")
)

print("Rows:", df_usage.count())
df_usage.printSchema()

Rows: 120000
root
 |-- customer_id: long (nullable = true)
 |-- log_date: string (nullable = true)
 |-- monthly_balance: double (nullable = true)
 |-- num_products: long (nullable = true)
 |-- product_type: string (nullable = true)
 |-- usage_log_id: long (nullable = true)



In [119]:
# Check NULLs

df_usage.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_usage.columns
]).show()

+-----------+--------+---------------+------------+------------+------------+
|customer_id|log_date|monthly_balance|num_products|product_type|usage_log_id|
+-----------+--------+---------------+------------+------------+------------+
|          0|       0|              0|           0|           0|           0|
+-----------+--------+---------------+------------+------------+------------+



### Missing Values — Usage

The `usage` dataset contains **120,000 records**.

A NULL-value check was performed on all columns, and **no missing values were found**.

Therefore, no missing-value imputation or removal was required for this dataset.

In [112]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- geography: string (nullable = false)
 |-- is_active: boolean (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- date_opened: date (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- balance: double (nullable = false)
 |-- num_products: integer (nullable = false)
 |-- has_cr_card: boolean (nullable = true)
 |-- estimated_salary: double (nullable = true)
 |-- is_churned: boolean (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_batch_ts: timestamp (nullable = true)



In [113]:
df_offers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- offer_id: string (nullable = true)
 |-- offer_type: string (nullable = false)
 |-- accepted: boolean (nullable = true)
 |-- date_offered: date (nullable = true)
 |-- accepted_flag: string (nullable = false)
 |-- date_offered_key: date (nullable = true)



In [111]:
df_tickets.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- ticket_id: string (nullable = true)
 |-- issue_type: string (nullable = false)
 |-- severity: string (nullable = false)
 |-- resolution_time_hrs: double (nullable = false)



In [120]:
df_usage.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- log_date: string (nullable = true)
 |-- monthly_balance: double (nullable = true)
 |-- num_products: long (nullable = true)
 |-- product_type: string (nullable = true)
 |-- usage_log_id: long (nullable = true)

